# Colab GPU — Chroma 인제스트 (doctrine-rag-ollama)

로컬 도커와 **같은 DB**를 쓰려면 아래를 맞추세요.

- `chromadb==0.5.5` (프로젝트 `backend/requirements.txt`)
- `EMBEDDING_MODEL` (예: `BAAI/bge-m3`)
- CSV 경로: `data/chunks/{army,navy,air_force}/*.csv`

**사전 준비:** `backend` 폴더와 `data/chunks`를 Colab에 올립니다 (Drive 업로드·zip 해제·Git clone 중 택1).

## 1) GPU 확인

In [ ]:
!nvidia-smi

import torch
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## 2) 프로젝트 경로 설정

아래 `BACKEND_ROOT`만 환경에 맞게 수정하세요.

- Drive 사용 예: `/content/drive/MyDrive/doctrine-rag-ollama/backend`
- zip 해제 예: `/content/doctrine-rag-ollama/backend`

In [ ]:
import os
import sys

# ★ Colab에서 backend 폴더가 있는 경로로 수정
BACKEND_ROOT = "/content/doctrine-rag-ollama/backend"

assert os.path.isdir(BACKEND_ROOT), f"경로 없음: {BACKEND_ROOT}"
os.chdir(BACKEND_ROOT)
sys.path.insert(0, BACKEND_ROOT)

print("cwd:", os.getcwd())

## (선택) Google Drive 마운트

프로젝트를 Drive에 두었다면 이 셀을 먼저 실행한 뒤, 위 `BACKEND_ROOT`를 Drive 경로로 설정합니다.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# BACKEND_ROOT = "/content/drive/MyDrive/.../backend"  # 위 셀과 통합 시 수동으로 다시 설정
pass

## 3) 의존성 설치

Colab에 이미 `torch`가 있을 수 있습니다. `requirements.txt`의 `chromadb`, `sentence-transformers` 등을 맞춥니다.

In [ ]:
%pip install -q -r requirements.txt

## 4) 환경 변수 (로컬 `.env` / 도커와 동일 권장)

`CHROMA_DIR`은 **backend 기준 상대 경로** `chroma_db`를 쓰면, 인제스트 결과가 `backend/chroma_db`에 생성됩니다.

In [ ]:
import os

os.environ.setdefault("EMBEDDING_MODEL", "BAAI/bge-m3")
os.environ.setdefault("CHROMA_DIR", "chroma_db")
os.environ.setdefault("CHUNKS_DATA_DIR", "data/chunks")
os.environ.setdefault("INGEST_BATCH_SIZE", "64")
os.environ.setdefault("EMBEDDING_BATCH_SIZE", "32")

# HF 캐시 (선택)
os.environ.setdefault("HF_HOME", "/content/hf_cache")

for k in ("EMBEDDING_MODEL", "CHROMA_DIR", "CHUNKS_DATA_DIR"):
    print(k, "=", os.environ[k])

## 5) 데이터 존재 확인

`data/chunks/army`, `navy`, `air_force` 아래에 `*.csv`가 있어야 합니다.

In [ ]:
from pathlib import Path

chunks = Path("data/chunks")
for branch in ("army", "navy", "air_force"):
    d = chunks / branch
    csvs = list(d.glob("*.csv")) if d.is_dir() else []
    print(branch, "csv_count:", len(csvs), "path:", d.resolve())

## 6) 전체 리셋 후 재인제스트

**주의:** 기존 `chroma_db`를 비우고 세 군 컬렉션을 다시 채웁니다. 시간이 오래 걸릴 수 있습니다.

도커와 동일하게 쓰려면 완료 후 **`backend/chroma_db` 전체**를 로컬 프로젝트에 덮어씁니다.

In [ ]:
# config 로드 전에 env가 반영되도록, 필요 시 런타임 재시작 후 위 셀부터 다시 실행
import importlib

import config
importlib.reload(config)

from rag_service import full_reset_and_reingest

result = full_reset_and_reingest()
result

## 7) 결과 확인

In [ ]:
import vector_store
import config

for b in config.SERVICE_BRANCHES:
    col = config.COLLECTION_MAP[b]
    n = vector_store.collection_count(col)
    print(b, col, "chunks:", n)

## 8) `chroma_db` 압축 후 다운로드 (Colab)

생성된 폴더를 로컬 `doctrine-rag-ollama/backend/chroma_db`에 풀면 도커 마운트와 동일하게 사용됩니다.

In [ ]:
from google.colab import files
import shutil

zip_path = "/content/chroma_db_out.zip"
shutil.make_archive("/content/chroma_db_out", "zip", root_dir=".", base_dir="chroma_db")
files.download(zip_path)

---

### 로컬에 반영

1. `chroma_db_out.zip` 압축 해제
2. 내용을 `doctrine-rag-ollama/backend/chroma_db/` 에 넣기 (기존 폴더 백업 권장)
3. `docker compose up -d backend` 후 `GET /health`로 `vector_db` 카운트 확인

### Git으로 Colab에 가져오기 예시

```bash
!git clone https://github.com/YOUR_ORG/doctrine-rag-ollama.git /content/doctrine-rag-ollama
```

그 다음 `BACKEND_ROOT = "/content/doctrine-rag-ollama/backend"` 로 설정합니다.